In [ ]:
# OpenCV libraries
import numpy as np
import cv2
from IPython.display import display, Image

In [ ]:
# video color picker
import time
import numpy as np
import cv2
from IPython.display import display, Image

# 1. Open camera
got.open_camera()

# 2. Create a display window in Jupyter
dummy = np.zeros((240, 320, 3), np.uint8)
ok, jpg = cv2.imencode(".jpg", dummy)
handle = display(Image(data=jpg.tobytes()), display_id=True)

# 3. Show live video for 10 seconds
start_time = time.time()
duration = 10

while time.time() - start_time < duration:
    picture_raw = got.read_camera_data()
    if picture_raw is None:
        continue

    # Convert raw bytes to BGR image
    picture_1D = np.frombuffer(picture_raw, np.uint8)
    picture_bgr = cv2.imdecode(picture_1D, cv2.IMREAD_COLOR)
    if picture_bgr is None:
        continue

    # Convert BGR to HSV
    picture_hsv = cv2.cvtColor(picture_bgr, cv2.COLOR_BGR2HSV)

    # Get image size
    height = picture_hsv.shape[0]
    width = picture_hsv.shape[1]

    # Find center point
    center_x = width // 2
    center_y = height // 2

    # Read HSV value at center pixel
    h = int(picture_hsv[center_y, center_x, 0])
    s = int(picture_hsv[center_y, center_x, 1])
    v = int(picture_hsv[center_y, center_x, 2])

    # Draw center marker
    cv2.circle(picture_bgr, (center_x, center_y), 8, (0, 255, 0), 2)
    cv2.line(picture_bgr, (center_x - 15, center_y), (center_x + 15, center_y), (0, 255, 0), 1)
    cv2.line(picture_bgr, (center_x, center_y - 15), (center_x, center_y + 15), (0, 255, 0), 1)

    # Show HSV value on screen
    text = f"H={h}  S={s}  V={v}"
    cv2.putText(picture_bgr, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # Show countdown
    time_left = duration - int(time.time() - start_time)
    cv2.putText(picture_bgr, f"Time left: {time_left}s", (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    # Encode and update display
    ok, jpg = cv2.imencode(".jpg", picture_bgr)
    if ok:
        handle.update(Image(data=jpg.tobytes()))

    time.sleep(0.05)

print("Finished 10-second live color picker.")

In [ ]:
# blue cube recognition
def blue_cube_centralizing_approaching_Kp():
    # import libraries
    import time
    import numpy as np
    import cv2
    from IPython.display import display, Image

    # create color range
    lower_color = np.array([60,190,100])
    upper_color = np.array([140,255,180])

    # Kp values
    Kp_x = 0.08
    Kp_y = 0.3

    frame_center = 320
    target_w = 160
    dead_zone_x = 20
    dead_zone_y = 5

    # function 1: get the bgr image
    def picture_bgr():
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            return None
        return cv2.imdecode(np.frombuffer(picture_raw,np.uint8),cv2.IMREAD_COLOR)  

    # function 2: find color and return x,y,w,h
    def find_ball(frame):
        mask = cv2.inRange(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV),lower_color,upper_color)
        contours,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return None
        return cv2.boundingRect(max(contours,key=cv2.contourArea))

    # function 3: Kp move and draw the bounding box
    def move_and_draw(frame,x,y,w,h):
        center_x = x + w//2

        # calculate errors
        error_x = center_x - frame_center
        error_y = target_w - w

        # check: close enough and stop
        if abs(error_x) < dead_zone_x and abs(error_y) <dead_zone_y:
            got.mecanum_stop()
            return True
        
        #Kp output
        speed_x = int(Kp_x * error_x)
        speed_y = int(Kp_y * error_y)

        # clamp to safe speed range
        speed_x = max(min(speed_x, 25),-25)
        speed_y = max(min(speed_y, 25),-25)

        got.mecanum_move_xyz(speed_x, speed_y,0)

        #draw bounding box and debug info        
        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
        cv2.putText(frame,f"cx:{center_x},ex:{error_x},ey:{error_y}w:{w}",(x,y-10),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0),2)
        cv2.circle(frame,(center_x,y+h//2),5,(255,0,0),-1)

    # main program
    got.open_camera()
    ok,picture_jpg = cv2.imencode(".jpg",picture_bgr())
    handle = display(Image(picture_jpg.tobytes()),display_id = True)

    while True:
        frame = picture_bgr()
        if frame is None:
            continue

        result = find_ball(frame)
        if result:
            if move_and_draw(frame,*result):  # close enough and stop
                break
        else:
            got.mecanum_stop()
        
        ok,picture_jpg = cv2.imencode(".jpg",frame)
        if ok:
            handle.update(Image(picture_jpg.tobytes()))
        
        time.sleep(0.05)

In [ ]:
blue_cube_centralizing_approaching_Kp()
got.mecanum_move_speed_times(0,10,10,1)

In [ ]:
# red cube recognition
def red_cube_centralizing_approaching_Kp():
    # import libraries
    import time
    import numpy as np
    import cv2
    from IPython.display import display, Image

    # create color range
    lower_color = np.array([150,200,150])
    upper_color = np.array([200,255,200])
    
    # Kp values
    Kp_x = 0.08
    Kp_y = 0.3

    frame_center = 320
    target_w = 100
    dead_zone_x = 20
    dead_zone_y = 5

    # function 1: get the bgr image
    def picture_bgr():
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            return None
        return cv2.imdecode(np.frombuffer(picture_raw,np.uint8),cv2.IMREAD_COLOR)  

    # function 2: find color and return x,y,w,h
    def find_ball(frame):
        mask = cv2.inRange(cv2.cvtColor(frame, cv2.COLOR_BGR2HSV),lower_color,upper_color)
        contours,_ = cv2.findContours(mask, cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return None
        return cv2.boundingRect(max(contours,key=cv2.contourArea))

    # function 3: Kp move and draw the bounding box
    def move_and_draw(frame,x,y,w,h):
        center_x = x + w//2

        # calculate errors
        error_x = center_x - frame_center
        error_y = target_w - w

        # check: close enough and stop
        if abs(error_x) < dead_zone_x and abs(error_y) <dead_zone_y:
            got.mecanum_stop()
            return True
        
        #Kp output
        speed_x = int(Kp_x * error_x)
        speed_y = int(Kp_y * error_y)

        # clamp to safe speed range
        speed_x = max(min(speed_x, 10),-10)
        speed_y = max(min(speed_y, 10),-10)

        got.mecanum_move_xyz(speed_x, speed_y,0)

        #draw bounding box and debug info        
        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
        cv2.putText(frame,f"cx:{center_x},ex:{error_x},ey:{error_y}w:{w}",(x,y-10),cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0),2)
        cv2.circle(frame,(center_x,y+h//2),5,(255,0,0),-1)

    # main program
    got.open_camera()
    ok,picture_jpg = cv2.imencode(".jpg",picture_bgr())
    handle = display(Image(picture_jpg.tobytes()),display_id = True)

    while True:
        frame = picture_bgr()
        if frame is None:
            continue

        result = find_ball(frame)
        if result:
            if move_and_draw(frame,*result):  # close enough and stop
                break
        else:
            got.mecanum_stop()
        
        ok,picture_jpg = cv2.imencode(".jpg",frame)
        if ok:
            handle.update(Image(picture_jpg.tobytes()))
        
        time.sleep(0.05)

In [ ]:
blue_cube_centralizing_approaching_Kp()
time.sleep(0.5)
got.mecanum_move_speed_times(0,10,10,1)

In [ ]:
# move and face centralization approaching
def face_recog_approaching():
    import time
    import numpy as np
    import cv2
    from IPython.display import display, Image, clear_output

    got.open_camera()
    got.load_models(["face_recognition"])
    time.sleep(1)
    print("The camera is on.")

    while True:
        picture_raw = got.read_camera_data()
        if picture_raw is None:
            continue
        picture_1D = np.frombuffer(picture_raw,np.uint8)
        picture_bgr = cv2.imdecode(picture_1D,cv2.IMREAD_COLOR)  
        
        face_info = got.get_face_recognition_total_info()

        if face_info:
            name,face_x,face_y,face_height,face_width,face_area = face_info[0]
            face_x      = int(face_x)
            face_y      = int(face_y)
            face_height = int(face_height)
            face_width  = int(face_width)
            
            cv2.rectangle(picture_bgr, (face_x - face_width//2, face_y -face_height//2), (face_x + face_width//2, face_y +face_height//2), (0, 255, 0), 2)
            cv2.putText(picture_bgr, f"{name}  face_x:{face_x} face_width:{face_width}",
                        (face_x - face_width//2, face_y -face_height//2 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        ok,picture_jpg = cv2.imencode(".jpg",picture_bgr)
        clear_output(wait=True)
        display(Image(picture_jpg.tobytes()))
        time.sleep(0.01)

        face_info = got.get_face_recognition_total_info()
        
        if not face_info:
            got.mecanum_move_speed(0,10)
        elif face_info[0][1]>340:
            got.mecanum_move_xyz(8,5,0)
        elif face_info[0][1]<300:
            got.mecanum_move_xyz(-8,5,0)
        elif face_info[0][4]<80:
            got.mecanum_move_xyz(0,5,0)
        else:
            got.mecanum_stop()
            break

In [ ]:
face_recog_approaching()

## Final presentation

In [ ]:
# recognize the blue cube, capture it and deliver it to the customer.
import time
blue_cube_centralizing_approaching_Kp()
time.sleep(0.5)
got.mecanum_move_speed_times(0,10,10,1)
time.sleep(0.5)
got.mecanum_turn_speed_times(2,30,180,2)
time.sleep(0.5)
face_recog_approaching()
got.mecanum_move_speed_times(1,10,20,1)
got.mecanum_turn_speed_times(2,30,180,2)

# recognize the red cube, capture it and deliver it to the customer.
red_cube_centralizing_approaching_Kp()
time.sleep(0.5)
got.mecanum_move_speed_times(0,10,10,1)
time.sleep(0.5)
got.mecanum_turn_speed_times(2,30,180,2)
time.sleep(0.5)
face_recog_approaching()